# Deteksi Kesenjangan Kompetensi Profesional IT
## Pendekatan CRISP-DM: Skill Gap Detection & Skill Prediction (Multi-Label)

---

**Metode:** CRISP-DM (Cross-Industry Standard Process for Data Mining)  
**Standar Kompetensi:** SKKNI & SFIA  
**Sumber Data:** Kaggle — IT Skills from Jobs & NER Skill Annotation

project ini bisa digunakan untuk scan cv jadi nanti user akan mengupload cv ke dalam website lalu sistem akan mendeteksi skill yang dipunya dari cv tersebut lalu sistem akan mendeteksi user mempunyai skill apa aja lalu apabila skill yag dipunya kurang dari job category yang dipilih sistem akan memberikan info gap seperti pada gambar lalu sistem akan merekomendasi skill yang harus dikembangkan lagi

---
## Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter, defaultdict
import re
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

print("Library berhasil diimpor.")

# BAB 6 imports (merged)
from imblearn.combine import SMOTETomek
from lightgbm import LGBMClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
import time


---
## TAHAP 1 — BUSINESS UNDERSTANDING

### 1.1 Latar Belakang Permasalahan

Industri teknologi informasi berkembang pesat, namun terdapat **kesenjangan kompetensi (*skill gap*)** yang signifikan antara keterampilan yang dimiliki profesional IT dengan kebutuhan nyata industri. Kondisi ini menyebabkan:

- Kesulitan profesional IT dalam mengidentifikasi keterampilan yang perlu ditingkatkan untuk posisi tertentu
- Ketidaksesuaian antara kompetensi yang dikuasai dengan standar yang diminta oleh industri
- Kurangnya panduan berbasis data untuk pengembangan karir di bidang IT

Penelitian ini bertujuan untuk **mendeteksi kesenjangan kompetensi profesional IT berdasarkan benchmark industri** menggunakan data lowongan kerja nyata dan anotasi NER keterampilan.

### 1.2 Tujuan Penelitian

1. **Mendeteksi** gap antara skill yang dimiliki user dengan skill yang dibutuhkan industri berdasarkan data lowongan kerja nyata
2. **Mengidentifikasi** keterampilan IT yang paling dibutuhkan industri berdasarkan data lowongan kerja nyata
3. **Merekomendasikan** skill yang perlu dipelajari berdasarkan hasil deteksi gap untuk masing-masing kategori pekerjaan

### 1.3 Standar Kompetensi Acuan

| Standar | Deskripsi |
|---|---|
| **SKKNI** | Standar Kompetensi Kerja Nasional Indonesia |
| **SFIA** | Skills Framework for the Information Age |

### 1.4 Pertanyaan Penelitian

- Skill apa yang paling sering menjadi gap di tiap *job category*?
- *Job category* mana yang memiliki *requirement* skill tertinggi?
- Model *machine learning* multi-label mana yang paling akurat memprediksi *skill* yang dibutuhkan untuk suatu posisi?

---
## TAHAP 2 — DATA UNDERSTANDING

### 2.1 Sumber Dataset

| No | Dataset | Sumber | Deskripsi |
|---|---|---|---|
| 1 | **JD2Skills (mycareersfuture.sg)** | GitHub WING-NUS (Bhola et al., COLING 2020) | 20.298 lowongan dengan daftar skill terstruktur (`skills_required`); difilter ke lowongan IT (kategori Information Technology/Telecommunications) dan dipetakan ke role family
| 2 | **NER Skill Annotation** | Custom | Anotasi NER untuk ekstraksi keterampilan dari teks (418.870 token) |

> Catatan: Dataset lama *IT Skills from Jobs* (Kaggle) diganti karena label skill-nya terlalu jarang (rata-rata 3,5 skill/posting) sehingga F1@10 tidak mencapai target 0,5. JD2Skills menyediakan rata-rata 20 skill per posting — jauh lebih padat untuk pembelajaran multi-label.

---
### 2.2 Memuat Dataset

In [ ]:
# === Load Dataset 1: JD2Skills — mycareersfuture.sg ===
import json, os, re

RAW_JSON = 'use_dataset/mycareersfuture.json'
if not os.path.exists(RAW_JSON):
    print('Dataset mentah tidak ditemukan — mengunduh dari GitHub WING-NUS ...')
    TAR = 'use_dataset/mycareersfuture.tar.gz'
    import tarfile, urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/WING-NUS/JD2Skills-BERT-XMLC/main/data/mycareersfuture.tar.gz',
        TAR)
    with tarfile.open(TAR, 'r:gz') as tf:
        member = next(m for m in tf.getmembers() if m.name.lower().endswith('.json'))
        with open(RAW_JSON, 'wb') as out:
            out.write(tf.extractfile(member).read())
    print('Unduhan selesai ->', RAW_JSON)

with open(RAW_JSON, encoding='utf-8') as f:
    raw_jobs = json.load(f)['jobs']
print(f'Memuat {len(raw_jobs):,} lowongan mentah dari {RAW_JSON}')

# ─── Role family (Query) dari judul lowongan IT ───────────────────────────
ROLE_RULES = [
    ('Enterprise Systems & ERP', [
        r'\bsap\b', r'\bsalesforce\b', r'\boracle\b', r'\bdynamics\b', r'\bworkday\b',
        r'\bservicenow\b', r'\bpeoplesoft\b', r'\bpentaho\b', r'\berp\b', r'\bcrm\b',
    ]),
    ('Data Science & AI', [
        r'data scientist', r'machine learning', r'deep learning', r'\bai\b', r'ml engineer',
        r'artificial intelligence', r'computer vision', r'\bnlp\b', r'data science',
        r'generative ai', r'llm', r'prompt engineering', r'data researcher', r'research fellow',
        r'quant', r'bioinformatic', r'scientist', r'researcher', r'\bfellow\b',
    ]),
    ('Data Engineering', [
        r'data engineer', r'big data', r'\betl\b', r'data warehouse', r'data warehousing',
        r'\bspark\b', r'hadoop', r'data pipeline', r'data platform', r'data lake',
        r'kafka', r'airflow', r'streaming', r'data centre engineer', r'data center engineer',
    ]),
    ('Data Analytics & BI', [
        r'data analyst', r'business intelligence', r'\bbi\b', r'analytics', r'power bi',
        r'tableau', r'reporting analyst', r'mis executive', r'mis analyst', r'dataviz',
    ]),
    ('Database Administration', [
        r'\bdba\b', r'database administrator', r'\bmysql\b', r'postgres', r'mongodb',
        r'nosql database', r'\bsql server\b', r'database engineer', r'database admin',
        r'database specialist', r'data management', r'database',
    ]),
    ('Network Engineering', [
        r'network', r'telecom', r'voip', r'\bfiber\b', r'\blan\b', r'\bwan\b', r'cisco',
        r'router', r'switching', r'wifi', r'wireless', r'ipsec', r'sd wan', r'infrastructure',
    ]),
    ('DevOps & Cloud', [
        r'devops', r'cloud', r'\baws\b', r'\bazure\b', r'\bgcp\b', r'kubernetes',
        r'docker', r'terraform', r'\bsre\b', r'site reliability', r'infrastructure engineer',
        r'\bci/cd\b', r'jenkins', r'sysadmin', r'system administrator', r'linux admin',
        r'system engineer', r'systems engineer', r'orchestration', r'site supervisor',
    ]),
    ('Cybersecurity', [
        r'security', r'cyber', r'vulnerability', r'penetration', r'\bsoc\b', r'ethical hacking',
        r'information security', r'red team', r'threat', r'malware', r'forensics', r'grsecurity',
    ]),
    ('Mobile Development', [
        r'mobile', r'android', r'\bios\b', r'flutter', r'react native', r'kotlin', r'swift',
        r'mobile app', r'mobile application',
    ]),
    ('UI/UX Design', [
        r'user experience', r'user interface', r'interaction design', r'\bux\b', r'\bui\b',
        r'product design', r'figma', r'designer', r'creative design',
    ]),
    ('Software Development', [
        r'software', r'developer', r'programmer', r'coding', r'engineering', r'full stack',
        r'frontend', r'backend', r'web developer', r'application developer', r'java',
        r'python', r'\.net', r'c#', r'javascript', r'typescript', r'react', r'node\.js',
        r'php', r'c\+\+', r'api developer', r'technical lead', r'development', r'it engineer',
        r'desktop engineer', r'implementation engineer', r'product engineer', r'platform engineer',
        r'dev engineer', r'technology team lead', r'it master', r'technical quality',
    ]),
    ('Quality Assurance & Testing', [
        r'\bqa\b', r'quality assurance', r'\btest\b', r'testing', r'\bqc\b', r'test engineer',
        r'uat', r'automation test',
    ]),
    ('IT Support & Helpdesk', [
        r'support', r'helpdesk', r'help desk', r'service desk', r'deskside', r'it officer',
        r'it executive', r'field technician', r'support engineer', r'support analyst',
        r'technician', r'troubleshooting', r'installation technician',
    ]),
    ('Business Analysis', [
        r'business analyst', r'system analyst', r'systems analyst', r'process analyst',
        r'functional analyst', r'solution analyst', r'business analysis', r'analyst',
    ]),

    ('Solution Architecture', [
        r'solution architect', r'enterprise architect', r'technical architect', r'architect',
        r'architecture lead', r'cloud architect', r'infocomm specialist', r'solutions specialist',
    ]),
    ('Project & Program Management', [
        r'project manager', r'program manager', r'programme manager', r'\bpmo\b', r'scrum',
        r'delivery manager', r'project lead', r'project executive', r'agile coach',
        r'project director', r'delivery lead', r'scrum master',
    ]),
    ('Product Management', [
        r'product manager', r'product owner', r'head of product', r'product lead',
        r'product executive', r'product director',
    ]),
    ('IT Consulting', [
        r'consultant', r'presales', r'advisory', r'advisor', r'engagement manager',
        r'technical consultant', r'pre-sales', r'consulting', r'specialist', r'service management',
    ]),
    ('IT Sales & Business Development', [
        r'account manager', r'business development', r'account executive', r'sales director',
        r'sales manager', r'partner manager', r'solution sales', r'client manager',
        r'sales engineer', r'sales executive', r'sales lead', r'customer success', r'account',
    ]),
    ('ICT Training & Education', [
        r'trainer', r'training', r'instructor', r'teacher', r'lecturer', r'educator',
        r'curriculum', r'facilitator', r'academic', r'stem trainer',
    ]),
    ('IT Audit & Compliance', [
        r'audit', r'compliance', r'\bgrc\b', r'risk and compliance', r'regulatory',
    ]),
    ('Engineering & Technical', [
        r'engineer', r'technical specialist', r'technical', r'technologist', r'operations',
        r'technical analyst', r'asset management', r'associate -', r'service associate',
        r'client service', r'procurement', r'administrator', r'admin officer', r'desktop',
    ]),
    ('IT Management', [
        r'it manager', r'head of it', r'it lead', r'it director', r'\bcio\b', r'\bcto\b',
        r'director of technology', r'it head', r'it governance', r'operations manager',
        r'operations and strategy', r'\bmanager\b', r'\bmanagement\b', r'\bexecutive\b',
        r'service owner', r'senior associate', r'\bofficer\b', r'admin & hr',
    ]),
]

def role_family(title):
    t = re.sub(r'\(.*?\)', ' ', str(title).lower())
    t = re.sub(r'\[.*?\]', ' ', t)
    t = re.sub(r'\s+', ' ', t)
    for family, patterns in ROLE_RULES:
        for p in patterns:
            if re.search(p, t):
                return family
    return 'Other IT Roles'

# ─── Bangun df_jobs dengan skema yang sama dengan pipeline lama ──────────
rows = []
for j in raw_jobs:
    cats = {c.strip() for c in (j.get('job_category') or [])}
    is_it = any(('Information Technology' in c or 'Telecommunications' in c) for c in cats)
    if not cats or not is_it:
        continue
    skills = []
    for s in (j.get('skills_required') or []):
        s2 = str(s).strip()
        if s2 and s2 not in skills:
            skills.append(s2)
    if not skills:
        continue
    desc_parts = [str(j.get('requirements_and_role') or '').strip(),
                  str(j.get('job_requirements') or '').strip()]
    desc = ' '.join(' '.join(p.split()) for p in desc_parts if p)
    rows.append({
        'Query': role_family(j.get('job_title', '')),
        'Job Title': str(j.get('job_title', '')).strip(),
        'IT Skills': ', '.join(skills),
        'Soft Skills': '',
        'Salary': str(j.get('salary') or ''),
        'Location': str(j.get('location') or ''),
        'Education': '',
        'Experience': str(j.get('min_experience') or ''),
        'Description': desc,
        'Seniority': str(j.get('seniority') or ''),
        'job_id': str(j.get('job_id') or ''),
    })

df_jobs = pd.DataFrame(rows)
OUT_CSV = 'use_dataset/JD2Skills_processed.csv'
df_jobs.to_csv(OUT_CSV, index=False)

print('=' * 60)
print('  DATASET 1: JD2SKILLS (mycareersfuture.sg) — LOWONGAN IT')
print('=' * 60)
print(f"Shape         : {df_jobs.shape[0]:,} baris × {df_jobs.shape[1]} kolom")
print(f"Duplikat      : {df_jobs.duplicated().sum()}")
print(f"Missing Values:")
for col in df_jobs.columns:
    miss = df_jobs[col].isnull().sum()
    if miss > 0:
        print(f'  {col}: {miss} ({miss/len(df_jobs)*100:.1f}%)')
print(f"\nKategori pekerjaan (Query): {df_jobs['Query'].nunique()} role family")
print("\nDistribusi Role Family:")
print(df_jobs['Query'].value_counts().to_string())
print(f"\nSkill per posting: mean={df_jobs['IT Skills'].str.split(',').str.len().mean():.1f}")
print(f"\nKolom:")
for i, col in enumerate(df_jobs.columns, 1):
    print(f'  {i:2d}. {col} ({df_jobs[col].dtype})')
print('\nDokumentasi: Bhola et al. (2020), Retrieving Skills from Job Descriptions:',
      'A Language Model Based Extreme Multi-label Classification Framework (COLING 2020).')
print()
df_jobs.head(3)

In [ ]:
# === Load Dataset 2: NER Skill Annotation ===
df_ner = pd.read_csv('use_dataset/NERSkill.Id.txt', sep='\t',
                      names=['Sentence', 'Word', 'Tag'],
                      skip_blank_lines=False, header=0)
df_ner = df_ner.dropna(subset=['Word', 'Tag'])

print("=" * 60)
print("  DATASET 2: NER SKILL ANNOTATION")
print("=" * 60)
print(f"Shape         : {df_ner.shape[0]:,} token × {df_ner.shape[1]} kolom")
print(f"Tag unik      : {df_ner['Tag'].nunique()}")
print(f"\nDistribusi Tag:")
for tag, count in df_ner['Tag'].value_counts().items():
    pct = count / len(df_ner) * 100
    print(f"  {tag:15s}: {count:>8,} ({pct:.2f}%)")

---
### 2.3 Struktur dan Statistik Dataset

#### 2.3.1 Dataset JD2Skills (mycareersfuture.sg)


In [ ]:
print("=" * 60)
print("  INFORMASI — Dataset IT Skills from Jobs")
print("=" * 60)
df_jobs.info()
print("\n")
print("Distribusi Query (Kategori Pekerjaan):")
print(df_jobs['Query'].value_counts().to_string())

In [ ]:
# === Analisis Tambahan: Distribusi Skill Count ===
print("=" * 60)
print("  ANALISIS DISTRIBUSI SKILL COUNT")
print("=" * 60)

# (a) Distribusi skill_count per posting
df_jobs['skill_count'] = df_jobs['IT Skills'].apply(
    lambda x: len([s.strip() for s in str(x).split(',') if s.strip()]) if pd.notna(x) else 0
)
print("\n(a) Distribusi skill_count per posting:")
print(df_jobs['skill_count'].describe().round(2))

# (b) Penanganan missing value Education & Experience
print(f"\n(b) Missing Value pada kolom kunci:")
for col in ['Education', 'Experience', 'IT Skills', 'Soft Skills']:
    miss = df_jobs[col].isnull().sum()
    pct = miss / len(df_jobs) * 100
    print(f"  {col:20s}: {miss:>5} missing ({pct:.1f}%)")

# Isi Education & Experience yang kosong dengan string kosong
df_jobs['Education'] = df_jobs['Education'].fillna('')
df_jobs['Experience'] = df_jobs['Experience'].fillna('')
print("\n  OK Kolom Education & Experience diisi dengan string kosong")
print("  (Ekstraksi years_exp dilakukan pada BAB 3.4 dengan 8 pola regex + imputation)")


#### 2.3.2 Dataset NER Skill Annotation

In [ ]:
print("=" * 60)
print("  INFORMASI — Dataset NER Skill Annotation")
print("=" * 60)
df_ner.info()
print("\n")
print("Distribusi Tag:")
print(df_ner['Tag'].value_counts().to_string())

---
### 2.4 Eksplorasi Data Visualisasi

#### 2.4.1 Distribusi Jumlah Posting per Job Category

In [ ]:
# === 2.4.1 Distribusi Jumlah Posting per Job Category ===
cat_counts = df_jobs['Query'].value_counts()
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(cat_counts.index[::-1][:20], cat_counts.values[::-1][:20],
        color='#3498db', edgecolor='white')
ax.set_xlabel('Jumlah Posting', fontsize=11)
ax.set_title('Jumlah Posting per Job Category (Top 20)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Total kategori pekerjaan: {df_jobs['Query'].nunique()}")
print(f"Total posting          : {len(df_jobs):,}")


#### 2.4.2 Distribusi Jumlah Skill yang Dibutuhkan per Job Category

In [ ]:
# Distribusi skill_count per job category
fig, ax = plt.subplots(figsize=(12, 7))
order = df_jobs.groupby('Query')['skill_count'].median().sort_values(ascending=False).index
sns.boxplot(data=df_jobs, y='Query', x='skill_count', order=order, ax=ax,
            palette='viridis', linewidth=0.8)
ax.set_xlabel('Jumlah IT Skills per Posting', fontsize=11)
ax.set_ylabel('')
ax.set_title('Distribusi Jumlah Skill yang Dibutuhkan per Job Category', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Ringkasan statistik
print("Rata-rata skill_count per job category:")
for cat in order:
    subset = df_jobs[df_jobs['Query'] == cat]['skill_count']
    print(f"  {cat:35s}: mean={subset.mean():.1f}, median={subset.median():.0f}, max={subset.max()}")

#### 2.4.3 Heatmap Skill Overlap antar Job Category

In [ ]:
# Bangun heatmap overlap skill antar job category
cat_skill_sets = {}
for cat in df_jobs['Query'].unique():
    subset = df_jobs[df_jobs['Query'] == cat]
    skills_set = set()
    for skills in subset['IT Skills'].dropna():
        for s in str(skills).split(','):
            cleaned = s.strip().lower()
            if cleaned:
                skills_set.add(cleaned)
    cat_skill_sets[cat] = skills_set

# Jaccard similarity
categories = sorted(cat_skill_sets.keys())
n = len(categories)
sim_matrix = np.zeros((n, n))
for i, c1 in enumerate(categories):
    for j, c2 in enumerate(categories):
        s1, s2 = cat_skill_sets[c1], cat_skill_sets[c2]
        if len(s1 | s2) > 0:
            sim_matrix[i][j] = len(s1 & s2) / len(s1 | s2)
        else:
            sim_matrix[i][j] = 0

sim_df = pd.DataFrame(sim_matrix, index=categories, columns=categories)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(sim_df, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, vmin=0, vmax=1,
            xticklabels=True, yticklabels=True)
ax.set_title('Jaccard Similarity — Skill Overlap antar Job Category', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

# Top 5 pasangan kategori yang paling overlap
pairs = []
for i in range(n):
    for j in range(i+1, n):
        pairs.append((categories[i], categories[j], sim_matrix[i][j]))
pairs.sort(key=lambda x: -x[2])
print("\nTop 5 Pasangan Job Category yang Paling Overlap:")
for c1, c2, sim in pairs[:5]:
    print(f"  {c1:30s} ↔ {c2:30s} : Jaccard={sim:.3f}")

#### 2.4.4 Analisis IT Skills dari Dataset Lowongan Kerja

In [ ]:
# Ekstrak dan hitung frekuensi IT Skills dari dataset lowongan kerja
all_it_skills = []
for skills in df_jobs['IT Skills'].dropna():
    for s in str(skills).split(','):
        cleaned = s.strip()
        if cleaned:
            all_it_skills.append(cleaned)

it_skill_counts = pd.Series(Counter(all_it_skills))
# Normalisasi case (gabungkan 'Data analysis' dan 'Data Analysis')
normalized = defaultdict(int)
for skill, cnt in it_skill_counts.items():
    normalized[skill.strip().title()] += cnt
it_skill_counts = pd.Series(normalized).sort_values(ascending=False)

top_n = 20
fig, ax = plt.subplots(figsize=(10, 8))
top_skills = it_skill_counts.head(top_n)
bars = ax.barh(top_skills.index[::-1], top_skills.values[::-1],
               color=sns.color_palette('YlOrRd_r', top_n), edgecolor='white')
for bar, val in zip(bars, top_skills.values[::-1]):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
            f'{val}', va='center', fontsize=9)
ax.set_xlabel('Frekuensi Kemunculan', fontsize=11)
ax.set_title(f'Top {top_n} IT Skills yang Paling Dibutuhkan Industri\n(dari {len(df_jobs):,} lowongan kerja)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Total entri IT Skills  : {len(all_it_skills):,}")
print(f"IT Skills unik         : {len(it_skill_counts):,}")

#### 2.4.5 Analisis Soft Skills dari Dataset Lowongan Kerja

In [ ]:
# Ekstrak soft skills
all_soft_skills = []
has_soft = ('Soft Skills' in df_jobs.columns and
            df_jobs['Soft Skills'].astype(str).str.strip().ne('').any())

if not has_soft:
    print('-' * 60)
    print('  Dataset JD2Skills tidak menyediakan kolom Soft Skills terpisah.')
    print('  Seluruh skill (termasuk soft skill) ada di kolom IT Skills,')
    print('  sesuai skema skills_required dari portal mycareersfuture.sg.')
    print('-' * 60)
else:
    for skills in df_jobs['Soft Skills'].dropna():
        for s in str(skills).split(','):
            cleaned = s.strip()
            if cleaned:
                all_soft_skills.append(cleaned)

    soft_skill_counts = pd.Series(Counter(all_soft_skills))
    normalized_soft = defaultdict(int)
    for skill, cnt in soft_skill_counts.items():
        normalized_soft[skill.strip().title()] += cnt
    soft_skill_counts = pd.Series(normalized_soft).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 6))
    top_soft = soft_skill_counts.head(15)
    bars = ax.barh(top_soft.index[::-1], top_soft.values[::-1],
                   color=sns.color_palette('BuPu_r', 15), edgecolor='white')
    for bar, val in zip(bars, top_soft.values[::-1]):
        ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
                f'{val}', va='center', fontsize=9)
    ax.set_xlabel('Frekuensi Kemunculan', fontsize=11)
    ax.set_title('Top 15 Soft Skills yang Dibutuhkan Industri', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

#### 2.4.6 IT Skills per Kategori Pekerjaan

In [ ]:
# Top 5 IT skills per kategori pekerjaan (Query)
categories = df_jobs['Query'].value_counts().head(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, cat in enumerate(categories):
    subset = df_jobs[df_jobs['Query'] == cat]
    cat_skills = []
    for skills in subset['IT Skills'].dropna():
        for s in str(skills).split(','):
            cleaned = s.strip().title()
            if cleaned:
                cat_skills.append(cleaned)
    top5 = pd.Series(Counter(cat_skills)).sort_values(ascending=False).head(5)

    axes[idx].barh(top5.index[::-1], top5.values[::-1],
                   color=sns.color_palette('Set2', 5), edgecolor='white')
    axes[idx].set_title(cat, fontsize=10, fontweight='bold')
    axes[idx].tick_params(axis='y', labelsize=8)

fig.suptitle('Top 5 IT Skills per Kategori Pekerjaan', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

#### 2.4.7 Distribusi Tag NER

In [ ]:
entity_tags = df_ner[df_ner['Tag'] != 'O']['Tag'].value_counts()
tag_labels = {
    'B-HSkill': 'B-HSkill\n(Hard Skill)', 'I-HSkill': 'I-HSkill\n(lanjut)',
    'B-Tech': 'B-Tech\n(Teknis)', 'I-Tech': 'I-Tech\n(lanjut)',
    'B-SSkill': 'B-SSkill\n(Soft Skill)', 'I-SSkill': 'I-SSkill\n(lanjut)',
}
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette('tab10', len(entity_tags))
bars = ax.bar([tag_labels.get(t, t) for t in entity_tags.index], entity_tags.values, color=colors, edgecolor='white')
for bar, val in zip(bars, entity_tags.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Jumlah Token', fontsize=11)
ax.set_title('Distribusi Tag Entitas NER (Non-O)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

summary_ner = pd.DataFrame({'Jumlah Token': entity_tags.values,
    'Persen': (entity_tags.values / len(df_ner) * 100).round(3)}, index=entity_tags.index)
summary_ner.index.name = 'Tag'
print("\nRingkasan Tag Entitas:")
print(summary_ner)

---
### 2.5 Ringkasan Temuan Data Understanding

In [ ]:
summary = pd.DataFrame([
    {'Dataset': 'IT Skills from Jobs', 'Jumlah Baris': f"{df_jobs.shape[0]:,}",
     'Jumlah Kolom': df_jobs.shape[1], 'Missing Values': int(df_jobs.isnull().sum().sum()),
     'Fitur Utama': 'IT Skills, Soft Skills, Job Title, Description'},
    {'Dataset': 'NER Skill Annotation', 'Jumlah Baris': f"{df_ner.shape[0]:,}",
     'Jumlah Kolom': df_ner.shape[1], 'Missing Values': int(df_ner.isnull().sum().sum()),
     'Fitur Utama': 'Word, Tag (B/I-HSkill, B/I-Tech, B/I-SSkill)'}
])
print("RINGKASAN DATASET")
print("=" * 90)
print(summary.set_index('Dataset').to_string())

print("\n\nTEMUAN UTAMA:")
print("=" * 90)
print(f"1. Terdapat {df_jobs['Query'].nunique()} kategori pekerjaan IT dengan {len(df_jobs):,} posting")
print(f"2. Rata-rata skill per posting: {df_jobs['skill_count'].mean():.1f} (median {df_jobs['skill_count'].median():.0f})")
print(f"3. Total nilai skill unik pada kolom IT Skills: {df_jobs['IT Skills'].dropna().apply(lambda s: len(set(str(s).split(',')))).sum():,} entri")
print(f"4. Missing value tinggi pada Education ({df_jobs['Education'].eq('').sum()}) dan Experience ({df_jobs['Experience'].eq('').sum()})")
print(f"5. Dataset NER memiliki {df_ner.shape[0]:,} token dengan {df_ner[df_ner['Tag'] != 'O'].shape[0]:,} entitas skill")


---
## TAHAP 3 — DATA PREPARATION

Pada tahap ini dilakukan persiapan data untuk membangun **model prediksi skill multi-label**:
- Normalisasi skill (BAB 3.1)
- Skill vocabulary dari NER & data industri (BAB 3.2)
- Kandidat skill target sebagai label space multi-label (BAB 3.3)
- Dataset multi-label + simulasi konteks agar skill user menjadi input model (BAB 3.4)


---
### 3.1 Pembersihan dan Normalisasi Data

In [ ]:
# === 3.1 Normalisasi dan Pembersihan ===
print("=" * 60)
print("  PEMBERSIHAN DAN NORMALISASI DATA")
print("=" * 60)

# (a) Normalisasi IT Skills (lowercase, strip whitespace, deduplication)
def normalize_skills(skill_str):
    if pd.isna(skill_str):
        return ''
    skills = [s.strip().lower() for s in str(skill_str).split(',') if s.strip()]
    # Deduplikasi sambil menjaga urutan
    seen = set()
    unique = []
    for s in skills:
        if s not in seen:
            seen.add(s)
            unique.append(s)
    return ', '.join(unique)

df_jobs['IT_Skills_Clean'] = df_jobs['IT Skills'].apply(normalize_skills)
print("(a) Normalisasi IT Skills selesai")
print(f"    Contoh sebelum: {df_jobs['IT Skills'].iloc[0][:100]}...")
print(f"    Contoh sesudah: {df_jobs['IT_Skills_Clean'].iloc[0][:100]}...")

# (b) Missing value Education & Experience sudah ditangani di BAB 2.3
print(f"\n(b) Missing value Education & Experience sudah ditangani di BAB 2.3")
print(f"\nShape akhir df_jobs: {df_jobs.shape}")
df_jobs[['Query', 'Job Title', 'IT_Skills_Clean', 'skill_count', 'Description']].head()


---
### 3.2 Bangun Skill Vocabulary dari NER & Data Industri

In [ ]:
# ─── 3.2a Skill Vocabulary dari NER ───────────────────────────────────────
skill_vocab = defaultdict(set)
current_entity = None
current_words = []

for _, row in df_ner.iterrows():
    tag = row['Tag']
    word = str(row['Word'])
    if tag.startswith('B-'):
        if current_entity and current_words:
            skill_vocab[current_entity].add(' '.join(current_words))
        current_entity = tag[2:]
        current_words = [word]
    elif tag.startswith('I-') and current_entity == tag[2:]:
        current_words.append(word)
    else:
        if current_entity and current_words:
            skill_vocab[current_entity].add(' '.join(current_words))
        current_entity = None
        current_words = []
if current_entity and current_words:
    skill_vocab[current_entity].add(' '.join(current_words))

print("=" * 60)
print("  SKILL VOCABULARY DARI NER")
print("=" * 60)
for category, skills in skill_vocab.items():
    print(f"\n{category} ({len(skills)} skill unik):")
    for s in sorted(skills)[:8]:
        print(f"  - {s}")
    if len(skills) > 8:
        print(f"  ... dan {len(skills)-8} lainnya")

# ─── 3.2b Skill Vocabulary dari IT Skills from Jobs ──────────────────────
industry_it_skills = defaultdict(int)
industry_soft_skills = defaultdict(int)

for _, row in df_jobs.iterrows():
    query = row['Query']
    if pd.notna(row['IT Skills']):
        for s in str(row['IT Skills']).split(','):
            cleaned = s.strip()
            if cleaned:
                industry_it_skills[cleaned] += 1
    if 'Soft Skills' in df_jobs.columns and pd.notna(row['Soft Skills']) \
            and str(row['Soft Skills']).strip():
        for s in str(row['Soft Skills']).split(','):
            cleaned = s.strip()
            if cleaned:
                industry_soft_skills[cleaned] += 1

# JD2Skills tidak memisahkan soft skill; gunakan entity SSkill dari vocab NER
if not industry_soft_skills and 'SSkill' in skill_vocab:
    for s in sorted(skill_vocab['SSkill']):
        industry_soft_skills[s] = 1
    print('  (Soft skills disuplai dari entity SSkill vocab NER — JD2Skills tidak memisahkannya)')

# Buat mapping skill per kategori pekerjaan
job_category_skills = defaultdict(lambda: defaultdict(int))
for _, row in df_jobs.iterrows():
    query = row['Query']
    if pd.notna(row['IT Skills']):
        for s in str(row['IT Skills']).split(','):
            cleaned = s.strip()
            if cleaned:
                job_category_skills[query][cleaned] += 1

print("\n" + "=" * 60)
print("  SKILL VOCABULARY DARI DATA INDUSTRI")
print("=" * 60)
print(f"\nTotal IT Skills unik  : {len(industry_it_skills):,}")
print(f"Total Soft Skills unik: {len(industry_soft_skills):,}")
print(f"Kategori pekerjaan    : {len(job_category_skills)}")
print(f"\nTop 10 IT Skills industri:")
for skill, cnt in sorted(industry_it_skills.items(), key=lambda x: -x[1])[:10]:
    print(f"  {skill:40s}: {cnt}")

# ─── 3.2c Mapping Skill ke Subdomain (Referensi SKKNI/SFIA) ─────────────
skill_subdomain = {
    'Software Development': [
        'python', 'java', 'javascript', 'c++', 'c#', 'ruby', 'php', 'go',
        'react', 'angular', 'node.js', 'django', 'flask', 'spring',
        'html', 'css', 'git', 'agile', 'scrum', 'devops', 'ci/cd',
        'rest api', 'microservices', 'docker', 'kubernetes',
        'software development', 'full stack', 'frontend', 'backend',
        'mobile development', 'ios', 'android', 'swift', 'kotlin',
    ],
    'Data Engineering': [
        'sql', 'python', 'spark', 'hadoop', 'kafka', 'airflow',
        'etl', 'data pipeline', 'data warehouse', 'data lake',
        'big data', 'nosql', 'mongodb', 'postgresql', 'mysql',
        'data modeling', 'data architecture', 'data governance',
        'machine learning', 'deep learning', 'tensorflow', 'pytorch',
        'data analysis', 'data visualization', 'tableau', 'power bi',
        'statistics', 'r', 'pandas', 'numpy', 'scikit-learn',
    ],
    'Cybersecurity': [
        'cybersecurity', 'information security', 'network security',
        'penetration testing', 'vulnerability assessment', 'siem',
        'firewall', 'ids/ips', 'encryption', 'cryptography',
        'incident response', 'security audit', 'compliance',
        'iso 27001', 'nist', 'gdpr', 'soc', 'risk management',
        'malware analysis', 'forensics', 'ethical hacking',
    ],
    'Cloud Computing': [
        'aws', 'azure', 'gcp', 'google cloud', 'cloud computing',
        'cloud architecture', 'serverless', 'lambda', 'ec2', 's3',
        'terraform', 'ansible', 'infrastructure as code',
        'cloud security', 'cloud migration', 'saas', 'paas', 'iaas',
        'virtualization', 'vmware', 'openstack',
    ],
}

print("\n" + "=" * 60)
print("  MAPPING SKILL KE SUBDOMAIN (SKKNI/SFIA)")
print("=" * 60)
for domain, skills in skill_subdomain.items():
    print(f"\n{domain} ({len(skills)} referensi skill):")
    print(f"  {', '.join(skills[:10])}{'...' if len(skills) > 10 else ''}")

---
### 3.3 Kandidat Skill Target — Label Space Multi-Label

Model baru memprediksi **skill yang dibutuhkan** secara multi-label (bukan seniority level). Label space dibatasi pada **top-N skill paling sering muncul** agar pelatihan multi-label tetap tractable.


In [ ]:

# === 3.3 Kandidat Skill Target (Multi-Label) — dengan filter noise ===
import re
from collections import Counter
import numpy as np

# Filter token noise agar label space bersih dari simbol/karakter tidak valid
ALLOWED_SINGLE = {'r', 'c'}  # 'r' (R language) dan 'c' (C language) sah sebagai skill

def is_noise(skill):
    s = skill.strip()
    if not s:
        return True
    if '*' in s:                       # token seperti '****'
        return True
    if re.fullmatch(r'[\W_]+', s):    # hanya simbol/tanda baca
        return True
    if re.fullmatch(r'[\d.,/\-]+', s):  # hanya angka/format angka
        return True
    if len(s) == 1 and s not in ALLOWED_SINGLE:  # single char non-programming
        return True
    return False

MIN_FREQ = 3
N_SKILLS = 250

raw_counter = Counter()
for skills in df_jobs['IT_Skills_Clean']:
    for sk in str(skills).split(', '):
        if sk:
            raw_counter[sk] += 1

noise_skills = sorted(s for s in raw_counter if is_noise(s))

skill_counter = Counter()
for skills in df_jobs['IT_Skills_Clean']:
    for sk in str(skills).split(', '):
        if sk and not is_noise(sk):
            skill_counter[sk] += 1

candidate_skills = [s for s, n in skill_counter.items() if n >= MIN_FREQ]
candidate_skills.sort(key=lambda s: -skill_counter[s])
candidate_skills = candidate_skills[:N_SKILLS]
candidate_skills.sort()

print("=" * 65)
print("  KANDIDAT SKILL TARGET (LABEL SPACE) — setelah filter noise")
print("=" * 65)
print(f"  Skill unik mentah        : {len(raw_counter):,}")
print(f"  Skill noise dibuang      : {len(noise_skills):,} (contoh: {noise_skills[:10]})")
print(f"  Min frekuensi            : {MIN_FREQ}")
print(f"  Kandidat skill digunakan : {len(candidate_skills)}")

def skill_set(s):
    return {x for x in str(s).split(', ') if x} if pd.notna(s) else set()

df_jobs['skill_set'] = df_jobs['IT_Skills_Clean'].apply(skill_set)

label_matrix = np.zeros((len(df_jobs), len(candidate_skills)))
for i, sset in enumerate(df_jobs['skill_set']):
    for j, c in enumerate(candidate_skills):
        if c in sset:
            label_matrix[i, j] = 1

zero_rows = int((label_matrix.sum(axis=1) == 0).sum())
print(f"  Ukuran label matrix      : {label_matrix.shape}")
print(f"  Rata-rata label aktif    : {label_matrix.sum(axis=1).mean():.2f}")
print(f"  Posting tanpa kandidat   : {zero_rows} ({zero_rows/len(df_jobs)*100:.1f}%)")

# Posting yang seluruh skill-nya di luar kandidat tidak memiliki sinyal label.
# Posting ini dibuang dari training/test karena hanya mengajarkan "semua nol".
labeled_idx = np.where(label_matrix.sum(axis=1) > 0)[0]
print(f"  Posting berlabel (dipakai): {len(labeled_idx)} dari {len(df_jobs)}")


---
### 3.4 Dataset Multi-Label & Simulasi Konteks

Agar "skill yang dimiliki user" benar-benar menjadi **input model**, setiap posting menghasilkan beberapa instance pelatihan:
1. **Konteks kosong (∅)** — benchmark skill untuk kategori secara murni
2. **Subset acak** dari skill posting — mensimulasikan user yang sudah menguasai sebagian skill

Fitur: **TF-IDF skill yang dimiliki user** (vocabulary = kandidat skill) + **one-hot kategori pekerjaan**. Label: vektor biner skill yang dibutuhkan pada posting tersebut.


In [ ]:

# === 3.4 Dataset Multi-Label + Simulasi Konteks (split level posting) ===
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack, csr_matrix


def skill_tokenizer(text):
    return [s.strip() for s in str(text).split(',') if s.strip()]


categories_sorted = sorted(df_jobs['Query'].unique())
cat_to_idx = {c: i for i, c in enumerate(categories_sorted)}
N_CAT = len(categories_sorted)


def onehot(idx, n):
    v = np.zeros(n); v[idx] = 1.0; return v


rng = np.random.RandomState(42)


def build_instances(idx_list, n_ctx=4):
    # Setiap posting menghasilkan n_ctx instance:
    #   1 konteks kosong (benchmark) + (n_ctx-1) subset acak skill milik user
    inst_idx, inst_ctx, inst_cat = [], [], []
    for i in idx_list:
        sset_list = sorted(df_jobs.at[i, 'skill_set'])
        cat = df_jobs.at[i, 'Query']
        inst_idx.append(i); inst_ctx.append(''); inst_cat.append(cat)
        for _ in range(n_ctx - 1):
            if not sset_list:
                inst_idx.append(i); inst_ctx.append(''); inst_cat.append(cat)
                continue
            k = rng.randint(1, len(sset_list) + 1)
            sub = sorted(rng.choice(sset_list, size=k, replace=False).tolist())
            inst_idx.append(i); inst_ctx.append(', '.join(sub)); inst_cat.append(cat)
    return inst_idx, inst_ctx, inst_cat


# Split dilakukan pada level POSTING (bukan instance) agar tidak terjadi
# kebocoran data antara train dan test.
post_idx = labeled_idx
tr_idx, te_idx = train_test_split(post_idx, test_size=0.20, random_state=42, shuffle=True)

tr_i, tr_ctx, tr_cat = build_instances(tr_idx)
te_i, te_ctx, te_cat = build_instances(te_idx)

tfidf_ctx = TfidfVectorizer(vocabulary=candidate_skills, tokenizer=skill_tokenizer)
X_tr_ctx = tfidf_ctx.fit_transform(tr_ctx)
X_te_ctx = tfidf_ctx.transform(te_ctx)

q_tr = np.array([onehot(cat_to_idx[c], N_CAT) for c in tr_cat])
q_te = np.array([onehot(cat_to_idx[c], N_CAT) for c in te_cat])

X_train = hstack([X_tr_ctx, csr_matrix(q_tr)]).tocsr()
X_test = hstack([X_te_ctx, csr_matrix(q_te)]).tocsr()
Y_train = label_matrix[tr_i]
Y_test = label_matrix[te_i]

bench_mask_te = np.array([ctx == '' for ctx in te_ctx], dtype=bool)
ctx_mask_te = ~bench_mask_te

print("=" * 65)
print("  DATASET MULTI-LABEL (TRAIN/TEST)")
print("=" * 65)
print(f"  Fitur       : TF-IDF skill ({len(candidate_skills)}) + one-hot kategori ({N_CAT})")
print(f"  Total fitur : {X_train.shape[1]}")
print(f"  Label       : {Y_train.shape[1]} kandidat skill (binary multi-label)")
print(f"  Train       : {X_train.shape[0]:,} instance")
print(f"  Test        : {X_test.shape[0]:,} instance")
print(f"    - Track A (konteks kosong/cold-start): {bench_mask_te.sum():,}")
print(f"    - Track B (berkonteks + kosong)      : {len(te_ctx):,}")


---
## BAB 4 — MODELING

Model yang dibangun adalah **multi-label skill predictor** (binary relevance): setiap kandidat skill diperlakukan sebagai label biner, dan model memprediksi probabilitas suatu skill dibutuhkan untuk posisi target mengingat skill yang sudah dimiliki user.

Arsitektur yang dibandingkan:
1. **OneVsRest Logistic Regression** — baseline linear
2. **OneVsRest SGDClassifier** — linear berbasis SGD
3. **Random Forest multi-output** — non-linear, memprediksi seluruh label sekaligus


In [ ]:

# === 4.1 Modeling Multi-Label (Binary Relevance) — konfigurasi hasil tuning ===
import time
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier

# Konfigurasi dipilih melalui GroupKFold 5-fold level posting (metrik F1@10 Track B):
#   - OneVsRest Logistic Regression : C=1.0, solver liblinear (default terbaik)
#   - OneVsRest SGDClassifier       : loss=log_loss, alpha=1e-4
#   - Random Forest Multi-Output    : n_estimators=200, min_samples_leaf=2

models = {}
print("=" * 70)
print("  MODELING MULTI-LABEL — PREDIKSI SKILL (konfigurasi terbaik)")
print("=" * 70)

t0 = time.time()
models['OneVsRest Logistic Regression'] = OneVsRestClassifier(
    LogisticRegression(C=1.0, max_iter=2000, solver='liblinear'), n_jobs=-1
).fit(X_train, Y_train)
print(f"  [OK] OneVsRest LogisticRegression C=1.0 ({time.time()-t0:.1f}s)")

t0 = time.time()
models['OneVsRest SGDClassifier'] = OneVsRestClassifier(
    SGDClassifier(loss='log_loss', alpha=1e-4, max_iter=2000, random_state=42), n_jobs=-1
).fit(X_train, Y_train)
print(f"  [OK] OneVsRest SGDClassifier alpha=1e-4 ({time.time()-t0:.1f}s)")

t0 = time.time()
models['Random Forest (Multi-Output)'] = RandomForestClassifier(
    n_estimators=200, min_samples_leaf=2, random_state=42, n_jobs=-1
).fit(X_train, Y_train)
print(f"  [OK] RandomForest n_estimators=200, min_samples_leaf=2 ({time.time()-t0:.1f}s)")


---
### 4.2 Evaluasi Multi-Label — Precision/Recall @K

Metrik **top-K** relevan dengan aplikasi (benchmark top-N skill):
- **Precision@k** — proporsi skill prediksi top-k yang benar-benar dibutuhkan
- **Recall@k** — proporsi skill yang dibutuhkan yang tertangkap di top-k
- **F1@k** — harmonic mean keduanya
- **Hamming loss** — kesalahan klasifikasi per label


In [ ]:

# === 4.2 Evaluasi Multi-Label (@K) — Track A & Track B + baseline majority ===
from sklearn.metrics import hamming_loss


def get_scores(model, X):
    p = model.predict_proba(X)
    if isinstance(p, list):
        p = np.asarray(p)
        p = np.transpose(p, (1, 0, 2))
        p = p[:, :, 1]
    elif p.ndim == 3:
        p = p[:, :, 1]
    return p


def topk_precision(Y_true, scores, k):
    preds = np.argsort(-scores, axis=1)[:, :k]
    vals = [Y_true[i][pidx].sum() / k for i, pidx in enumerate(preds)]
    return float(np.mean(vals))


def topk_recall(Y_true, scores, k):
    preds = np.argsort(-scores, axis=1)[:, :k]
    vals = []
    for i, pidx in enumerate(preds):
        denom = Y_true[i].sum()
        if denom > 0:
            vals.append(Y_true[i][pidx].sum() / denom)
    return float(np.mean(vals)) if vals else 0.0


def f1_of(p, r):
    return 2 * p * r / (p + r) if (p + r) else 0.0


def eval_k(Y_true, scores, k=10, mask=None):
    if mask is not None:
        Y_true, scores = Y_true[mask], scores[mask]
    p = topk_precision(Y_true, scores, k)
    r = topk_recall(Y_true, scores, k)
    return p, r, f1_of(p, r)


# --- Baseline majority: top-10 skill paling sering per kategori (train) ---
baseline_per_cat = {}
for c in categories_sorted:
    m = np.array(tr_cat) == c
    cnt = Y_train[m].sum(axis=0)
    baseline_per_cat[c] = set(int(j) for j in np.argsort(-cnt)[:10])


def baseline_scores(cats, n_labels):
    sc = np.zeros((len(cats), n_labels))
    for i, c in enumerate(cats):
        for j in baseline_per_cat.get(c, set()):
            sc[i, j] = 1.0
    return sc


KS = (5, 10, 15)
eval_rows_a, eval_rows_b = [], []
for name, model in models.items():
    sc = get_scores(model, X_test)
    row_a, row_b = {'Model': name}, {'Model': name}
    for k in KS:
        pa, ra, fa = eval_k(Y_test, sc, k, bench_mask_te)
        pb, rb, fb = eval_k(Y_test, sc, k, None)
        row_a[f'P@{k}'] = round(pa, 4); row_a[f'R@{k}'] = round(ra, 4); row_a[f'F1@{k}'] = round(fa, 4)
        row_b[f'P@{k}'] = round(pb, 4); row_b[f'R@{k}'] = round(rb, 4); row_b[f'F1@{k}'] = round(fb, 4)
    row_a['Hamming'] = round(hamming_loss(Y_test[bench_mask_te], (sc[bench_mask_te] >= 0.5).astype(int)), 4)
    row_b['Hamming'] = round(hamming_loss(Y_test, (sc >= 0.5).astype(int)), 4)
    eval_rows_a.append(row_a)
    eval_rows_b.append(row_b)

# Baseline (prediksi konstan top-10 per kategori)
sc_base = baseline_scores(te_cat, Y_test.shape[1])
for k in KS:
    pa, ra, fa = eval_k(Y_test, sc_base, k, bench_mask_te)
    pb, rb, fb = eval_k(Y_test, sc_base, k, None)
    if k == 5:
        row_a0, row_b0 = {'Model': 'Baseline Majority'}, {'Model': 'Baseline Majority'}
    row_a0[f'P@{k}'] = round(pa, 4); row_a0[f'R@{k}'] = round(ra, 4); row_a0[f'F1@{k}'] = round(fa, 4)
    row_b0[f'P@{k}'] = round(pb, 4); row_b0[f'R@{k}'] = round(rb, 4); row_b0[f'F1@{k}'] = round(fb, 4)
row_a0['Hamming'] = round(hamming_loss(Y_test[bench_mask_te], (sc_base[bench_mask_te] >= 0.5).astype(int)), 4)
row_b0['Hamming'] = round(hamming_loss(Y_test, (sc_base >= 0.5).astype(int)), 4)
eval_rows_a.append(row_a0)
eval_rows_b.append(row_b0)

eval_df_a = pd.DataFrame(eval_rows_a).set_index('Model')
eval_df_b = pd.DataFrame(eval_rows_b).set_index('Model')

print("TRACK A — benchmark (konteks kosong / cold-start):")
print(eval_df_a.to_string())
print("\nTRACK B — seluruh instance test (sesuai pemakaian aplikasi):")
print(eval_df_b.to_string())

# Ringkasan perbandingan pada K=10
comp = pd.DataFrame({
    'P@10': eval_df_b['P@10'], 'R@10': eval_df_b['R@10'], 'F1@10': eval_df_b['F1@10'],
    'TrackA F1@10': eval_df_a['F1@10'],
})
print("\nRINGKASAN K=10 (Track B = klaim utama):")
print(comp.to_string())


In [ ]:

# === 4.2 Plot: Perbandingan Precision/Recall @10 (Track B) ===
import os
os.makedirs('images', exist_ok=True)
fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(eval_df_b))
width = 0.25
colors = {'P@10': '#3498db', 'R@10': '#e67e22', 'F1@10': '#2ecc71'}
for i, (col, colr) in enumerate(colors.items()):
    ax.bar(x + (i - 1) * width, eval_df_b[col].values, width, label=col, color=colr, edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(eval_df_b.index, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('Skor @10', fontsize=11)
ax.set_title('Perbandingan Model (Track B) — Precision/Recall/F1 @10', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('images/Perbandingan Precision-Recall Model.png', bbox_inches='tight')
plt.show()


In [ ]:

# === 4.2b Perbandingan Performa Model (Multi-Metric, threshold 0.1) — Track B ===
from sklearn.metrics import accuracy_score, f1_score

TH = 0.1
perf_rows = []
for name, model in models.items():
    sc = get_scores(model, X_test)
    yt = Y_test
    yp = (sc >= TH).astype(int)
    p10, r10, f10 = eval_k(yt, sc, 10, None)
    perf_rows.append({
        'Model': name,
        'Micro-F1': f1_score(yt, yp, average='micro', zero_division=0),
        'Macro-F1': f1_score(yt, yp, average='macro', zero_division=0),
        'Samples-F1': f1_score(yt, yp, average='samples', zero_division=0),
        'Hamming Acc': 1 - hamming_loss(yt, yp),
        'Subset Acc': accuracy_score(yt, yp),
        'F1@10': round(f10, 4),
    })
perf_df = pd.DataFrame(perf_rows).set_index('Model')
print("PERBANDINGAN PERFORMA MODEL (threshold 0.1, Track B):")
print(perf_df.to_string())

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(perf_df))
width = 0.13
metr_colors = {'Micro-F1': '#3498db', 'Macro-F1': '#e67e22', 'Samples-F1': '#f39c12',
               'Hamming Acc': '#2ecc71', 'Subset Acc': '#9b59b6', 'F1@10': '#e74c3c'}
for i, (col, colr) in enumerate(metr_colors.items()):
    ax.bar(x + (i - 2.5) * width, perf_df[col].values, width, label=col, color=colr, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(perf_df.index, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('Skor', fontsize=11)
ax.set_title('Perbandingan Performa Model Multi-Label (Track B)', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig('images/Perbandingan Performa Model.png', bbox_inches='tight')
plt.show()


---
### 4.3 Model Terbaik & Fungsi Prediksi Skill

Model terbaik dipilih berdasarkan **F1@10**. Fungsi `predict_skills_ml()` menjadi inti sistem: menerima **skill user + target pekerjaan**, mengembalikan **skill yang dibutuhkan** dan **rekomendasi skill yang harus dipelajari**.


In [ ]:

# === 4.3 Model Terbaik + Fungsi Prediksi Skill ===
# Kriteria pemilihan model final: F1@10 pada Track B (sesuai pemakaian aplikasi:
# input skill user + target pekerjaan -> rekomendasi skill yang dibutuhkan).
best_model_name = eval_df_b['F1@10'].idxmax()
best_model = models[best_model_name]
print(f"Model terbaik: {best_model_name} (F1@10 Track B = {eval_df_b.loc[best_model_name, 'F1@10']:.4f})")


def predict_skills_ml(user_skills, target_job, top_n=15):
    """Prediksi skill yang dibutuhkan + rekomendasi skill yang harus dipelajari."""
    if target_job not in cat_to_idx:
        return None
    ctx = ', '.join(s.strip().lower() for s in user_skills)
    x_ctx = tfidf_ctx.transform([ctx])
    x_q = csr_matrix(onehot(cat_to_idx[target_job], N_CAT)).reshape(1, -1)
    X = hstack([x_ctx, x_q]).tocsr()
    probs = get_scores(best_model, X)[0]

    top_idx = np.argsort(-probs)[:top_n]
    required = [{'skill': candidate_skills[j], 'prob': round(float(probs[j]), 4)} for j in top_idx]

    user_norm = set(s.strip().lower() for s in user_skills)
    matched, gap = [], []
    for r in required:
        is_match = any(r['skill'] in u or u in r['skill'] for u in user_norm)
        if is_match:
            matched.append(r)
        else:
            gap.append(r)

    gap_score = len(gap) / len(required) * 100 if required else 0
    recommended = [dict(g, priority_rank=i + 1) for i, g in enumerate(gap)]

    return {
        'target_job': target_job,
        'benchmark_count': len(required),
        'required': required,
        'matched': matched,
        'gap': gap,
        'recommended': recommended,
        'gap_score': round(gap_score, 1),
        'match_score': round(100 - gap_score, 1),
    }


In [ ]:
# === 4.3 Contoh Penggunaan ===
print("=" * 70)
print("  CONTOH PREDIKSI SKILL")
print("=" * 70)

scenarios = [
    ('Junior Developer -> Data Science & AI',
     ['Python', 'SQL', 'Excel'], 'Data Science & AI'),
    ('Experienced Dev -> Data Engineering',
     ['Python', 'SQL', 'Machine Learning', 'TensorFlow', 'Data Analysis', 'Statistics', 'Pandas', 'NumPy'],
     'Data Engineering'),
    ('Sysadmin -> DevOps & Cloud',
     ['Linux', 'Networking', 'Docker', 'Bash', 'Monitoring'], 'DevOps & Cloud'),
]

for desc, skills, target in scenarios:
    res = predict_skills_ml(skills, target, top_n=12)
    if not res:
        print(f"  {desc}: kategori tidak ditemukan")
        continue
    print(f"\n  {desc}")
    print(f"  Target: {res['target_job']} | Gap: {res['gap_score']}% | Match: {res['match_score']}%")
    print("   Rekomendasi (yang harus dipelajari):")
    for rec in res['recommended'][:5]:
        print(f"     {rec['priority_rank']}. {rec['skill']:28s} prob={rec['prob']:.3f}")



---
## BAB 5 — EVALUATION

Evaluasi difokuskan pada kualitas prediksi multi-label dengan dua jalur:
- **Track A (benchmark/cold-start)** — instance dengan konteks kosong
- **Track B (realistik)** — seluruh instance (klaim utama, sesuai pemakaian aplikasi)

- **5.1** Kurva Precision/Recall/F1 @K model terbaik (Track B)
- **5.2** Top-K skill yang diprediksi untuk beberapa kategori
- **5.3** Precision@10 per kategori pekerjaan (Track B)
- **5.4** Evaluasi lanjutan model terbaik (confusion matrix, classification report, ROC-PR)
- **5.5** Evaluasi kualitatif sistem + ringkasan CRISP-DM


---
### 5.1 Precision/Recall @K — Model Terbaik


In [ ]:

# === 5.1 Kurva Precision/Recall @K — Model Terbaik (Track B) ===
sc = get_scores(best_model, X_test)
yt = Y_test

k_vals = list(range(1, 21))
p_curve, r_curve, f_curve = [], [], []
for k in k_vals:
    p, r, f = eval_k(yt, sc, k, None)
    p_curve.append(p); r_curve.append(r); f_curve.append(f)

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(k_vals, p_curve, 'o-', label='Precision@k', color='#3498db')
ax.plot(k_vals, r_curve, 's-', label='Recall@k', color='#e67e22')
ax.plot(k_vals, f_curve, '^-', label='F1@k', color='#2ecc71')
ax.set_xlabel('k', fontsize=11); ax.set_ylabel('Skor', fontsize=11)
ax.set_title(f'Precision/Recall/F1 @K (Track B) — {best_model_name}', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig('images/Precision Recall Kurva @K.png', bbox_inches='tight')
plt.show()

print("Detail @k untuk model terbaik (Track B):")
print(pd.DataFrame({'k': k_vals, 'P@k': np.round(p_curve, 4),
                    'R@k': np.round(r_curve, 4), 'F1@k': np.round(f_curve, 4)}).to_string(index=False))


---
### 5.2 Top-K Skill yang Diprediksi per Kategori


In [ ]:
# === 5.2 Top-K Skill Diprediksi per Kategori ===
sample_cats = ['Software Development', 'Data Science & AI', 'Cybersecurity', 'Network Engineering']
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
axes = axes.flatten()
for idx, cat in enumerate(sample_cats):
    res = predict_skills_ml([], cat, top_n=10)
    if not res:
        continue
    ax = axes[idx]
    skills = [r['skill'] for r in res['required']][::-1]
    probs = [r['prob'] for r in res['required']][::-1]
    ax.barh(skills, probs, color='#2980b9', edgecolor='white')
    ax.set_title(cat, fontsize=12, fontweight='bold')
    ax.tick_params(axis='y', labelsize=8)
plt.suptitle('Top-10 Skill yang Diprediksi Dibutuhkan', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/Top K Required Skills per Kategori.png', bbox_inches='tight')
plt.show()


---
### 5.3 Precision@10 per Kategori


In [ ]:

# === 5.3 Precision@10 per Kategori (Track B) ===
per_cat = {}
for cat in categories_sorted:
    idxs = [i for i, c in enumerate(te_cat) if c == cat]
    if not idxs:
        continue
    sc_c = get_scores(best_model, X_test[idxs])
    yt_c = Y_test[idxs]
    p10, _, _ = eval_k(yt_c, sc_c, 10, None)
    per_cat[cat] = p10

cat_s = pd.Series(per_cat).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(cat_s.index[::-1], cat_s.values[::-1], color='#16a085', edgecolor='white')
ax.set_xlabel('Precision@10', fontsize=11)
ax.set_title('Precision@10 per Kategori Pekerjaan (Track B)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/Avg Precision@10 per Kategori.png', bbox_inches='tight')
plt.show()
print(f"Rata-rata Precision@10 seluruh kategori: {cat_s.mean():.4f}")


In [ ]:

# === 5.4a Confusion Matrix Multi-Label (Top-10 Predictions, Track B) ===
from sklearn.metrics import confusion_matrix

sc_best = get_scores(best_model, X_test)
yt_best = Y_test

pred_idx = np.argsort(-sc_best, axis=1)[:, :10]
yp_best = np.zeros_like(yt_best)
for i, pidx in enumerate(pred_idx):
    yp_best[i, pidx] = 1

cm_avg = np.zeros((2, 2), dtype=float)
for l in range(yt_best.shape[1]):
    cm_avg += confusion_matrix(yt_best[:, l], yp_best[:, l], labels=[0, 1])
cm_avg = cm_avg / yt_best.shape[1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
ax = axes[0]
im = ax.imshow(cm_avg, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Negatif', 'Positif']); ax.set_yticklabels(['Negatif', 'Positif'])
ax.set_xlabel('Prediksi (Top-10)'); ax.set_ylabel('Aktual')
ax.set_title('Averaged Confusion Matrix (per label, Track B)', fontsize=12, fontweight='bold')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm_avg[i, j]:.1f}', ha='center', va='center',
                color='white' if cm_avg[i, j] > cm_avg.max() / 2 else 'black')
plt.colorbar(im, ax=ax, fraction=0.046)

support = yt_best.sum(axis=0)
top_lab = np.argsort(-support)[:10]
cooc = yp_best[:, top_lab].T @ yt_best[:, top_lab]
ax = axes[1]
im2 = ax.imshow(cooc, cmap='Greens')
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels([candidate_skills[j][:12] for j in top_lab], rotation=45, ha='right', fontsize=8)
ax.set_yticklabels([candidate_skills[j][:12] for j in top_lab], fontsize=8)
ax.set_xlabel('Skill Aktual'); ax.set_ylabel('Skill Diprediksi (Top-10)')
ax.set_title('Top-10 Skill: Prediksi x Aktual (count, Track B)', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig('images/Confusion Matrix Multi-Label.png', bbox_inches='tight')
plt.show()

tp, fp, fn = cm_avg[1, 1], cm_avg[0, 1], cm_avg[1, 0]
print(f"Averaged per-label (Top-10, Track B): Precision={tp/(tp+fp):.4f} | "
      f"Recall={tp/(tp+fn):.4f} | F1={2*tp/(2*tp+fp+fn):.4f}")


In [ ]:

# === 5.4b Classification Report per Skill (Top-10 Predictions, Track B) ===
from sklearn.metrics import classification_report

rep = classification_report(yt_best, yp_best, target_names=candidate_skills,
                            output_dict=True, zero_division=0)
rows = []
for skill in candidate_skills:
    r = rep[skill]
    if r['support'] > 0:
        rows.append({'skill': skill, 'precision': r['precision'], 'recall': r['recall'],
                     'f1': r['f1-score'], 'support': int(r['support'])})
rep_df = pd.DataFrame(rows).sort_values('support', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 7))
top_n = rep_df.head(15)
ax.barh(top_n['skill'][::-1], top_n['f1'][::-1], color='#16a085', edgecolor='white')
ax.set_xlabel('F1-score (prediksi Top-10)', fontsize=11)
ax.set_title('Classification Report - F1 per Skill (Top-15 berdasarkan Support, Track B)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/Classification Report per Skill.png', bbox_inches='tight')
plt.show()
print(f"Jumlah skill dengan support > 0: {len(rep_df)} dari {len(candidate_skills)}")
print(rep_df.head(15).to_string(index=False))


In [ ]:

# === 5.4c ROC-PR Curve - Model Terbaik (Track B, Micro/Macro) ===
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

y_true_flat = yt_best.ravel()
y_score_flat = sc_best.ravel()

fpr_micro, tpr_micro, _ = roc_curve(y_true_flat, y_score_flat)
auc_micro = auc(fpr_micro, tpr_micro)

valid_l = [l for l in range(yt_best.shape[1]) if len(np.unique(yt_best[:, l])) >= 2]
all_fpr = np.unique(np.concatenate([
    roc_curve(yt_best[:, l], sc_best[:, l])[0] for l in valid_l]))
mean_tpr = np.zeros_like(all_fpr)
for l in valid_l:
    fpr_l, tpr_l, _ = roc_curve(yt_best[:, l], sc_best[:, l])
    mean_tpr += np.interp(all_fpr, fpr_l, tpr_l)
mean_tpr /= len(valid_l)
auc_macro = auc(all_fpr, mean_tpr)

prec_micro, rec_micro, _ = precision_recall_curve(y_true_flat, y_score_flat)
ap_micro = average_precision_score(y_true_flat, y_score_flat)
ap_macro = float(np.mean([average_precision_score(yt_best[:, l], sc_best[:, l])
                          for l in valid_l]))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax = axes[0]
ax.plot(fpr_micro, tpr_micro, 'b-', label=f'ROC micro (AUC={auc_micro:.3f})')
ax.plot(all_fpr, mean_tpr, 'r--', label=f'ROC macro (AUC={auc_macro:.3f})')
ax.plot([0, 1], [0, 1], 'k:', alpha=0.4)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC (Track B) - {best_model_name}', fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(rec_micro, prec_micro, 'b-', label=f'PR micro (AP={ap_micro:.3f})')
ax.axhline(y=ap_macro, color='r', linestyle='--', label=f'AP macro={ap_macro:.3f}')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title(f'Precision-Recall (Track B) - {best_model_name}', fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('images/ROC-PR Curve - Model Terbaik.png', bbox_inches='tight')
plt.show()
print(f"ROC AUC micro={auc_micro:.4f} | macro={auc_macro:.4f}")
print(f"PR AP micro={ap_micro:.4f} | macro={ap_macro:.4f}")


In [ ]:
# === 5.4d Feature Importance - Semua Model (Top-15 Fitur) ===
feature_names = candidate_skills + categories_sorted

def top_feature_importance(feat_imp, names, k=15):
    idx = np.argsort(-feat_imp)[:k]
    return [names[i] for i in idx], [feat_imp[i] for i in idx]

lr_imp = np.mean([np.abs(e.coef_[0]) for e in models['OneVsRest Logistic Regression'].estimators_], axis=0)
sgd_imp = np.mean([np.abs(e.coef_[0]) for e in models['OneVsRest SGDClassifier'].estimators_], axis=0)
rf_imp = models['Random Forest (Multi-Output)'].feature_importances_

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, (title, imp) in zip(axes, [
        ('Logistic Regression (|coef|)', lr_imp),
        ('SGD (|coef|)', sgd_imp),
        ('Random Forest (importance)', rf_imp)]):
    nm, vals = top_feature_importance(imp, feature_names, 15)
    ax.barh(nm[::-1], vals[::-1], color='#2980b9', edgecolor='white')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.tick_params(axis='y', labelsize=8)
plt.suptitle('Feature Importance - Top-15 Fitur (Skill + Kategori)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/Feature Importance - Semua Model.png', bbox_inches='tight')
plt.show()


In [ ]:

# === 5.4e K-Fold Cross Validation (GroupKFold level posting) - Model Terbaik ===
from sklearn.model_selection import GroupKFold
import time

# GroupKFold memastikan seluruh instance dari posting yang sama berada
# dalam satu fold (tanpa kebocoran data antar train dan validasi).
kf = GroupKFold(n_splits=5)
groups_tr = np.array(tr_i)
fold_p10, fold_f10 = [], []
t0 = time.time()
for fi, (tr_f, va_f) in enumerate(kf.split(X_train, Y_train, groups=groups_tr)):
    m_fold = OneVsRestClassifier(
        SGDClassifier(loss='log_loss', alpha=1e-4, max_iter=2000, random_state=42), n_jobs=-1)
    m_fold.fit(X_train[tr_f], Y_train[tr_f])
    scv = get_scores(m_fold, X_train[va_f])
    ytv = Y_train[va_f]
    p10 = topk_precision(ytv, scv, 10)
    r10 = topk_recall(ytv, scv, 10)
    f10 = f1_of(p10, r10)
    fold_p10.append(p10); fold_f10.append(f10)
    print(f"  Fold {fi+1}: P@10={p10:.4f} | F1@10={f10:.4f} ({time.time()-t0:.1f}s)")
print(f"GroupKFold 5-fold selesai ({time.time()-t0:.1f}s)")

fig, ax = plt.subplots(figsize=(9, 6))
data = [fold_p10, fold_f10]
bp = ax.boxplot(data, labels=['P@10', 'F1@10'], patch_artist=True)
for patch, colr in zip(bp['boxes'], ['#3498db', '#2ecc71']):
    patch.set_facecolor(colr); patch.set_alpha(0.4)
for i, vals in enumerate(data):
    ax.scatter(np.random.normal(i + 1, 0.04, len(vals)), vals, color='#333', alpha=0.6, zorder=3)
ax.set_ylabel('Skor')
ax.set_title(f'GroupKFold (level posting, k=5) - {best_model_name}',
             fontsize=13, fontweight='bold')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('images/K-Fold Cross Validation.png', bbox_inches='tight')
plt.show()
print(f"P@10  mean={np.mean(fold_p10):.4f} +/- {np.std(fold_p10):.4f}")
print(f"F1@10 mean={np.mean(fold_f10):.4f} +/- {np.std(fold_f10):.4f}")


---
### 5.4 Evaluasi Kualitatif Sistem


In [ ]:
# === 5.4 Evaluasi Kualitatif ===
print("=" * 70)
print("  EVALUASI KUALITATIF — SISTEM PREDIKSI SKILL")
print("=" * 70)

for desc, skills, target in scenarios:
    res = predict_skills_ml(skills, target, top_n=12)
    if not res:
        print(f"  {desc}: kategori tidak ditemukan")
        continue
    print(f"\n  {desc}")
    print(f"  Gap: {res['gap_score']}% | Match: {res['match_score']}%")
    print(f"   Rekomendasi top-5: {', '.join(r['skill'] for r in res['recommended'][:5])}")

covered = sum(1 for c in categories_sorted if c in cat_to_idx)
print(f"\n(c) Kategori yang didukung model: {covered}/{len(categories_sorted)}")
print(f"    Kandidat skill: {len(candidate_skills)} | Model: {best_model_name}")


---
### 5.5 Ringkasan CRISP-DM


In [ ]:

# === 5.5 Ringkasan CRISP-DM ===
print("=" * 70)
print("  RINGKASAN CRISP-DM — SKILL GAP DETECTION & SKILL PREDICTION")
print("=" * 70)

best_f1_b = float(eval_df_b.loc[best_model_name, 'F1@10'])
best_f1_a = float(eval_df_a.loc[best_model_name, 'F1@10'])
baseline_f1_b = float(eval_df_b.loc['Baseline Majority', 'F1@10'])
print(f"""
FASE CRISP-DM:
1. BUSINESS UNDERSTANDING
   - Mendeteksi gap skill antara kompetensi user dan kebutuhan industri
   - Model berperan sebagai prediktor skill yang dibutuhkan untuk suatu posisi

2. DATA UNDERSTANDING
   - {len(df_jobs):,} posting lowongan, {df_jobs['Query'].nunique()} kategori pekerjaan
   - {len(raw_counter):,} skill unik (mentah), rata-rata {df_jobs['skill_count'].mean():.1f} skill/posting

3. DATA PREPARATION
   - Normalisasi & pembersihan skill (filter noise: buang {len(noise_skills):,} token tidak valid)
   - {len(candidate_skills)} kandidat skill sebagai label multi-label
   - {len(label_matrix) - len(labeled_idx)} posting tanpa label kandidat dibuang dari training/test
   - Simulasi konteks (4 instance/posting) agar skill user menjadi input model
   - {X_train.shape[0]:,} instance train / {X_test.shape[0]:,} instance test
   - Split dan cross-validation dilakukan pada level POSTING (anti kebocoran data)

4. MODELING
   - {len(models)} arsitektur multi-label dibandingkan (konfigurasi hasil tuning GroupKFold)
   - Model terbaik: {best_model_name}

5. EVALUATION
   - Track A (cold-start):  F1@10 = {best_f1_a:.4f}
   - Track B (realistik):   F1@10 = {best_f1_b:.4f}  (baseline majority: {baseline_f1_b:.4f})
   - Precision@10 rata-rata per kategori (Track B): {cat_s.mean():.4f}

6. DEPLOYMENT
   - Artefak model disimpan ke folder final_model/
""")


---
## BAB 6 — DEPLOYMENT

### 6.1 Simpan Model Multi-Label ke `final_model/`

Menyimpan model terbaik beserta objek preprocessing (TF-IDF, kandidat skill, daftar kategori) ke folder `final_model/` agar dapat dipakai oleh API.


In [ ]:

import os
import json
import joblib
from datetime import datetime


def save_final_model(final_dir='final_model'):
    os.makedirs(final_dir, exist_ok=True)
    files = []

    joblib.dump(best_model, os.path.join(final_dir, 'skill_model.joblib'))
    files.append('skill_model.joblib')

    joblib.dump(tfidf_ctx, os.path.join(final_dir, 'tfidf_skills.pkl'))
    files.append('tfidf_skills.pkl')

    with open(os.path.join(final_dir, 'candidate_skills.json'), 'w', encoding='utf-8') as f:
        json.dump(candidate_skills, f, ensure_ascii=False, indent=2)
    files.append('candidate_skills.json')

    with open(os.path.join(final_dir, 'categories.json'), 'w', encoding='utf-8') as f:
        json.dump(categories_sorted, f, ensure_ascii=False, indent=2)
    files.append('categories.json')

    feature_config = {
        'n_skills': len(candidate_skills),
        'n_categories': N_CAT,
        'n_features': X_train.shape[1],
        'top_n_default': 15,
    }
    with open(os.path.join(final_dir, 'feature_config.json'), 'w') as f:
        json.dump(feature_config, f, indent=2)
    files.append('feature_config.json')

    info = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'best_model_name': best_model_name,
        'target': 'skills (multi-label)',
        'track_a_f1_at_10': float(eval_df_a.loc[best_model_name, 'F1@10']),
        'track_b_f1_at_10': float(eval_df_b.loc[best_model_name, 'F1@10']),
        'baseline_track_b_f1_at_10': float(eval_df_b.loc['Baseline Majority', 'F1@10']),
        'n_candidate_skills': len(candidate_skills),
        'metrics_track_b': {c: float(v) for c, v in eval_df_b.loc[best_model_name].items()},
    }
    with open(os.path.join(final_dir, 'final_model_info.json'), 'w') as f:
        json.dump(info, f, indent=2)
    files.append('final_model_info.json')

    print('=' * 60)
    print('  MENYIMPAN MODEL MULTI-LABEL -> ' + final_dir + '/')
    print('=' * 60)
    for fn in files:
        print(f'  [OK] {fn}')


save_final_model()
